# DS1/GSE103224 — build the raw-count AnnData object

This notebook reconstructs the eight-sample GSE103224 count object from the
GEO supplementary files matching:

`GSM*_PJ*.filtered.matrix.txt.gz`

Each source file is a headerless, tab-separated gene-by-cell matrix in which:

1. column 1 is the versioned Ensembl gene identifier;
2. column 2 is the gene symbol;
3. remaining columns are raw integer counts for individual cells.

The parser reads the wide matrices in row chunks to avoid loading an entire
sample into a dense pandas object at once. Synthetic cell identifiers are
generated because the GEO files do not contain cell-barcode headers.

## Expected result

- 8 samples
- 23,793 cells
- 60,725 Ensembl features

## Default paths

Input: `data/raw/GSE103224/`

Output:
`data/interim/DS1_GSE103224/GSE103224_all_samples_inner.h5ad`

The paths may be overridden with `GLIOMA_PROJECT_ROOT`,
`GSE103224_RAW_DIR`, and `GSE103224_INTERIM_H5AD`.

> **Runtime patch v2:** GSE103224 files contain the same Ensembl feature
> set but not necessarily the same row order or display metadata.
> This version aligns every sample to the first sample's Ensembl-ID order,
> audits metadata differences, and verifies that all eight RAW files remain
> byte-for-byte unchanged during execution.

> **AnnData 0.12 compatibility patch:** backed `AnnData` objects are
> explicitly closed with `check.file.close()` rather than used as
> context managers.

In [1]:
import gc
import gzip
import hashlib
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_34732\1124959783.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_34732\1124959783.py:19: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE103224_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE103224",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_INTERIM_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS1_GSE103224"
        / "GSE103224_all_samples_inner.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS1_GSE103224" / "00a_build"
)

CHUNK_ROWS = int(os.environ.get("GSE103224_CHUNK_ROWS", "250"))

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    "GSM2758471": ("PJ016", 3085),
    "GSM2758472": ("PJ017", 1261),
    "GSM2758473": ("PJ018", 2197),
    "GSM2758474": ("PJ025", 5924),
    "GSM2758475": ("PJ030", 3097),
    "GSM2758476": ("PJ032", 1377),
    "GSM2758477": ("PJ035", 3768),
    "GSM2940098": ("PJ048", 3084),
}
EXPECTED_N_GENES = 60_725
EXPECTED_TOTAL_CELLS = 23_793

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)
print("Chunk rows:", CHUNK_ROWS)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"RAW directory not found: {RAW_DIR}\n"
        "Place the eight GEO matrix files there or set GSE103224_RAW_DIR."
    )

Project root: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas
RAW directory: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\raw\GSE103224
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS1_GSE103224\GSE103224_all_samples_inner.h5ad
Chunk rows: 250


In [3]:
FILE_PATTERN = re.compile(
    r"^(GSM\d+)_(PJ\d+)\.filtered\.matrix\.txt\.gz$"
)


def discover_input_files(raw_dir: Path) -> list[dict]:
    records = []
    for path in sorted(raw_dir.glob("GSM*_*.filtered.matrix.txt.gz")):
        match = FILE_PATTERN.match(path.name)
        if match is None:
            continue
        geo_sample_id, sample_id = match.groups()
        records.append(
            {
                "path": path,
                "geo_sample_id": geo_sample_id,
                "sample_id": sample_id,
            }
        )

    observed = {
        record["geo_sample_id"]: record["sample_id"] for record in records
    }
    expected = {
        geo_sample_id: values[0]
        for geo_sample_id, values in EXPECTED_FILES.items()
    }

    if observed != expected:
        missing = sorted(set(expected) - set(observed))
        extra = sorted(set(observed) - set(expected))
        mismatched = {
            key: (expected.get(key), observed.get(key))
            for key in set(expected) & set(observed)
            if expected[key] != observed[key]
        }
        raise ValueError(
            "The discovered GSE103224 files do not match the expected "
            f"eight-sample set. Missing={missing}, extra={extra}, "
            f"mismatched={mismatched}"
        )

    return records


def sha256_file(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


input_records = discover_input_files(RAW_DIR)

raw_manifest_before = pd.DataFrame(
    [
        {
            "geo_sample_id": record["geo_sample_id"],
            "sample_id": record["sample_id"],
            "filename": record["path"].name,
            "expected_cells": EXPECTED_FILES[record["geo_sample_id"]][1],
            "size_bytes": record["path"].stat().st_size,
            "sha256": sha256_file(record["path"]),
        }
        for record in input_records
    ]
)
raw_manifest_before.to_csv(
    AUDIT_DIR / "GSE103224_RAW_manifest_before.tsv",
    sep="\t",
    index=False,
)
raw_manifest_before

,geo_sample_id,sample_id,filename,expected_cells,size_bytes,sha256
0,GSM2758471,PJ016,GSM2758471_PJ016.filtered.matrix.txt.gz,3085,10956162,f4820eb240f57d18c030e21028d25ef2990b6f38020606...
1,GSM2758472,PJ017,GSM2758472_PJ017.filtered.matrix.txt.gz,1261,3666807,b869230c724aebd5d991647e8fc83c864adc4fdf0e3f28...
2,GSM2758473,PJ018,GSM2758473_PJ018.filtered.matrix.txt.gz,2197,6245711,e122d83884f082a6933fbedfd1c61efdb59b90148344a2...
3,GSM2758474,PJ025,GSM2758474_PJ025.filtered.matrix.txt.gz,5924,11872832,df1689dd393f6681a9a7ef1f76ff226df78c80f1c67afc...
4,GSM2758475,PJ030,GSM2758475_PJ030.filtered.matrix.txt.gz,3097,6093087,4c086091590ad5c5add59e424506dd13448597af5dfcbe...
5,GSM2758476,PJ032,GSM2758476_PJ032.filtered.matrix.txt.gz,1377,3340062,7aa23a26b3c77c3a46bd61cd2c9ed4cfea2afe4c6ee16d...
6,GSM2758477,PJ035,GSM2758477_PJ035.filtered.matrix.txt.gz,3768,6371341,b4773644dc790efefe64c1d1d4078505126c8410803923...
7,GSM2940098,PJ048,GSM2940098_PJ048.filtered.matrix.txt.gz,3084,6472089,26e10deeab7d2d8cb979bcb4cfb52a03c2da7bd7cc004a...


In [4]:
def _numeric_count_array(frame: pd.DataFrame, source: Path) -> np.ndarray:
    values = frame.to_numpy(copy=False)

    if not np.issubdtype(values.dtype, np.number):
        frame = frame.apply(pd.to_numeric, errors="raise")
        values = frame.to_numpy(copy=False)

    if np.issubdtype(values.dtype, np.floating):
        if not np.isfinite(values).all():
            raise ValueError(f"Non-finite count value detected in {source}")
        if not np.equal(values, np.floor(values)).all():
            raise ValueError(f"Non-integer count value detected in {source}")

    if np.any(values < 0):
        raise ValueError(f"Negative count value detected in {source}")

    maximum = values.max(initial=0)
    if maximum > np.iinfo(np.int32).max:
        raise OverflowError(
            f"A count in {source} exceeds the int32 range: {maximum}"
        )

    return values.astype(np.int32, copy=False)


def read_geo_matrix_chunked(
    path: Path,
    geo_sample_id: str,
    sample_id: str,
    chunk_rows: int = 250,
) -> ad.AnnData:
    gene_ids_versioned = []
    gene_symbols = []
    matrix_blocks = []
    n_cells = None

    reader = pd.read_csv(
        path,
        sep="\t",
        header=None,
        compression="gzip",
        chunksize=chunk_rows,
        low_memory=False,
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        if chunk.shape[1] < 3:
            raise ValueError(
                f"{path.name}, chunk {chunk_number}: expected at least "
                "three columns."
            )

        current_n_cells = chunk.shape[1] - 2
        if n_cells is None:
            n_cells = current_n_cells
        elif current_n_cells != n_cells:
            raise ValueError(
                f"Inconsistent cell-column count in {path.name}: "
                f"{current_n_cells} versus {n_cells}"
            )

        gene_ids_versioned.extend(chunk.iloc[:, 0].astype(str).tolist())
        gene_symbols.extend(chunk.iloc[:, 1].astype(str).tolist())

        counts = _numeric_count_array(chunk.iloc[:, 2:], path)
        matrix_blocks.append(sp.csr_matrix(counts.T))

        del chunk, counts
        gc.collect()

    if n_cells is None or not matrix_blocks:
        raise ValueError(f"No matrix rows were read from {path}")

    X = sp.hstack(matrix_blocks, format="csr").astype(np.int32, copy=False)
    del matrix_blocks
    gc.collect()

    ensembl_no_version = pd.Series(gene_ids_versioned).str.replace(
        r"\.\d+$", "", regex=True
    )

    if ensembl_no_version.duplicated().any():
        duplicates = (
            ensembl_no_version[ensembl_no_version.duplicated(keep=False)]
            .drop_duplicates()
            .head(10)
            .tolist()
        )
        raise ValueError(
            "Removing Ensembl version suffixes produced duplicate "
            f"feature identifiers in {path.name}: {duplicates}"
        )

    obs_names = pd.Index(
        [f"{sample_id}_cell{i:06d}" for i in range(n_cells)],
        name="cell_id",
    )
    obs = pd.DataFrame(index=obs_names)
    obs["sample_id"] = sample_id
    obs["batch"] = sample_id
    obs["geo_sample_id"] = geo_sample_id
    obs["geo_id"] = "GSE103224"
    obs["core_dataset"] = "DS1_GSE103224"

    var = pd.DataFrame(
        {
            "ensembl_id_version": gene_ids_versioned,
            "gene_symbol": gene_symbols,
        },
        index=pd.Index(ensembl_no_version, name="ensembl_id"),
    )

    sample = ad.AnnData(X=X, obs=obs, var=var)
    sample.uns["source_file"] = path.name
    sample.uns["source_geo_accession"] = "GSE103224"

    return sample

In [5]:
sample_objects = []
sample_summary_rows = []
reference_var = None
reference_gene_order = None
metadata_conflict_rows = []

for record in input_records:
    geo_sample_id = record["geo_sample_id"]
    sample_id = record["sample_id"]
    expected_cells = EXPECTED_FILES[geo_sample_id][1]

    print(f"Reading {record['path'].name} ...")
    sample = read_geo_matrix_chunked(
        path=record["path"],
        geo_sample_id=geo_sample_id,
        sample_id=sample_id,
        chunk_rows=CHUNK_ROWS,
    )

    if sample.n_obs != expected_cells:
        raise AssertionError(
            f"{sample_id}: expected {expected_cells} cells, "
            f"observed {sample.n_obs}"
        )
    if sample.n_vars != EXPECTED_N_GENES:
        raise AssertionError(
            f"{sample_id}: expected {EXPECTED_N_GENES} genes, "
            f"observed {sample.n_vars}"
        )
    if not sample.var_names.is_unique:
        raise AssertionError(
            f"{record['path'].name}: feature identifiers are not unique."
        )

    current_var = sample.var[
        ["ensembl_id_version", "gene_symbol"]
    ].copy()

    if reference_var is None:
        reference_var = current_var.copy()
        reference_gene_order = reference_var.index.copy()
    else:
        missing_ids = reference_gene_order.difference(sample.var_names)
        extra_ids = sample.var_names.difference(reference_gene_order)

        if len(missing_ids) or len(extra_ids):
            mismatch_table = pd.DataFrame(
                {
                    "status": (
                        ["missing_from_current"] * len(missing_ids)
                        + ["extra_in_current"] * len(extra_ids)
                    ),
                    "ensembl_id": list(missing_ids) + list(extra_ids),
                    "source_file": record["path"].name,
                }
            )
            mismatch_table.to_csv(
                AUDIT_DIR
                / f"{geo_sample_id}_{sample_id}_gene_set_mismatch.tsv",
                sep="\t",
                index=False,
            )
            raise ValueError(
                f"{record['path'].name} does not contain the same "
                "60,725 Ensembl identifiers as the first sample. "
                f"Missing={len(missing_ids)}, extra={len(extra_ids)}. "
                "See the mismatch audit table."
            )

        # The GEO files contain the same feature set, but row order and/or
        # display metadata can differ. Align every sample to the first
        # sample's Ensembl-ID order before concatenation.
        current_aligned = current_var.reindex(reference_gene_order)

        for column in ("ensembl_id_version", "gene_symbol"):
            reference_values = (
                reference_var[column].astype("string").fillna("")
            )
            current_values = (
                current_aligned[column].astype("string").fillna("")
            )
            differing = current_values.ne(reference_values)

            if differing.any():
                for ensembl_id in reference_gene_order[differing]:
                    metadata_conflict_rows.append(
                        {
                            "source_file": record["path"].name,
                            "ensembl_id": ensembl_id,
                            "field": column,
                            "reference_value": reference_values.loc[
                                ensembl_id
                            ],
                            "current_value": current_values.loc[
                                ensembl_id
                            ],
                        }
                    )

        sample = sample[:, reference_gene_order].copy()

        if not sample.var_names.equals(reference_gene_order):
            raise AssertionError(
                f"{record['path'].name}: feature reordering failed."
            )

        # Use one consistent feature annotation after auditing differences.
        sample.var = reference_var.copy()

    sample_summary_rows.append(
        {
            "geo_sample_id": geo_sample_id,
            "sample_id": sample_id,
            "cells": sample.n_obs,
            "genes": sample.n_vars,
            "total_counts": int(sample.X.sum()),
            "maximum_count": int(sample.X.max()),
            "source_file": record["path"].name,
        }
    )
    sample_objects.append(sample)
    print(sample)

sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(
    AUDIT_DIR / "GSE103224_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)

metadata_conflicts = pd.DataFrame(
    metadata_conflict_rows,
    columns=[
        "source_file",
        "ensembl_id",
        "field",
        "reference_value",
        "current_value",
    ],
)
metadata_conflicts.to_csv(
    AUDIT_DIR / "GSE103224_feature_metadata_conflicts.tsv",
    sep="\t",
    index=False,
)

print(
    "Feature metadata differences audited:",
    metadata_conflicts.shape[0],
)
sample_summary

Reading GSM2758471_PJ016.filtered.matrix.txt.gz ...
AnnData object with n_obs × n_vars = 3085 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_version', 'gene_symbol'
    uns: 'source_file', 'source_geo_accession'
Reading GSM2758472_PJ017.filtered.matrix.txt.gz ...
AnnData object with n_obs × n_vars = 1261 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_version', 'gene_symbol'
    uns: 'source_file', 'source_geo_accession'
Reading GSM2758473_PJ018.filtered.matrix.txt.gz ...
AnnData object with n_obs × n_vars = 2197 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_version', 'gene_symbol'
    uns: 'source_file', 'source_geo_accession'
Reading GSM2758474_PJ025.filtered.matrix.txt.gz ...
AnnData object with n_obs × n_vars = 5924 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_ver

,geo_sample_id,sample_id,cells,genes,total_counts,maximum_count,source_file
0,GSM2758471,PJ016,3085,60725,20020913,1034,GSM2758471_PJ016.filtered.matrix.txt.gz
1,GSM2758472,PJ017,1261,60725,3592776,759,GSM2758472_PJ017.filtered.matrix.txt.gz
2,GSM2758473,PJ018,2197,60725,6528367,371,GSM2758473_PJ018.filtered.matrix.txt.gz
3,GSM2758474,PJ025,5924,60725,15102424,459,GSM2758474_PJ025.filtered.matrix.txt.gz
4,GSM2758475,PJ030,3097,60725,4792046,334,GSM2758475_PJ030.filtered.matrix.txt.gz
5,GSM2758476,PJ032,1377,60725,2567725,606,GSM2758476_PJ032.filtered.matrix.txt.gz
6,GSM2758477,PJ035,3768,60725,4553505,131,GSM2758477_PJ035.filtered.matrix.txt.gz
7,GSM2940098,PJ048,3084,60725,6213163,510,GSM2940098_PJ048.filtered.matrix.txt.gz


In [6]:
adata_all = ad.concat(
    sample_objects,
    axis="obs",
    join="inner",
    merge="same",
    index_unique=None,
)
adata_all.var = reference_var.loc[adata_all.var_names].copy()

if not adata_all.obs_names.is_unique:
    raise AssertionError("Merged cell identifiers are not unique.")
if adata_all.n_obs != EXPECTED_TOTAL_CELLS:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_CELLS} total cells, "
        f"observed {adata_all.n_obs}"
    )
if adata_all.n_vars != EXPECTED_N_GENES:
    raise AssertionError(
        f"Expected {EXPECTED_N_GENES} genes, observed {adata_all.n_vars}"
    )
if not sp.issparse(adata_all.X):
    raise TypeError("The merged count matrix is unexpectedly dense.")

nonzero_values = adata_all.X.data
if np.any(nonzero_values < 0):
    raise AssertionError("Negative counts detected after concatenation.")
if not np.equal(nonzero_values, np.floor(nonzero_values)).all():
    raise AssertionError("Non-integer counts detected after concatenation.")

adata_all.uns["dataset_id"] = "DS1_GSE103224"
adata_all.uns["geo_accession"] = "GSE103224"
adata_all.uns["build_timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)
adata_all.uns["source_files"] = [
    record["path"].name for record in input_records
]

build_summary = {
    "dataset_id": "DS1_GSE103224",
    "geo_accession": "GSE103224",
    "n_samples": int(adata_all.obs["sample_id"].nunique()),
    "n_cells": int(adata_all.n_obs),
    "n_genes": int(adata_all.n_vars),
    "matrix_format": type(adata_all.X).__name__,
    "matrix_dtype": str(adata_all.X.dtype),
    "total_counts": int(adata_all.X.sum()),
    "maximum_count": int(adata_all.X.max()),
    "output_h5ad": str(OUTPUT_H5AD),
}

with open(
    AUDIT_DIR / "GSE103224_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

print(adata_all)
print(adata_all.obs["sample_id"].value_counts())
print(build_summary)

AnnData object with n_obs × n_vars = 23793 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_version', 'gene_symbol'
    uns: 'dataset_id', 'geo_accession', 'build_timestamp', 'source_files'
sample_id
PJ025    5924
PJ035    3768
PJ030    3097
PJ016    3085
PJ048    3084
PJ018    2197
PJ032    1377
PJ017    1261
Name: count, dtype: int64
{'dataset_id': 'DS1_GSE103224', 'geo_accession': 'GSE103224', 'n_samples': 8, 'n_cells': 23793, 'n_genes': 60725, 'matrix_format': 'csr_matrix', 'matrix_dtype': 'int32', 'total_counts': 63370919, 'maximum_count': 1034, 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\interim\\DS1_GSE103224\\GSE103224_all_samples_inner.h5ad'}


In [7]:
# Confirm that every RAW input still exists and is byte-for-byte unchanged.
raw_manifest_after_rows = []
for record in input_records:
    path = record["path"]
    if not path.exists():
        raise FileNotFoundError(
            f"RAW input disappeared during execution: {path}"
        )
    raw_manifest_after_rows.append(
        {
            "geo_sample_id": record["geo_sample_id"],
            "sample_id": record["sample_id"],
            "filename": path.name,
            "expected_cells": EXPECTED_FILES[
                record["geo_sample_id"]
            ][1],
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

raw_manifest_after = pd.DataFrame(raw_manifest_after_rows)
raw_manifest_after.to_csv(
    AUDIT_DIR / "GSE103224_RAW_manifest_after.tsv",
    sep="\t",
    index=False,
)

comparison_columns = [
    "geo_sample_id",
    "sample_id",
    "filename",
    "expected_cells",
    "size_bytes",
    "sha256",
]
if not raw_manifest_after[comparison_columns].equals(
    raw_manifest_before[comparison_columns]
):
    raise RuntimeError(
        "One or more GSE103224 RAW files changed during execution. "
        "The output will not be written."
    )

print("RAW input integrity verified: all 8 files are unchanged.")

temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_all.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

check = sc.read_h5ad(OUTPUT_H5AD, backed="r")
try:
    if check.shape != (EXPECTED_TOTAL_CELLS, EXPECTED_N_GENES):
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
finally:
    check.file.close()

print("Saved and validated:", OUTPUT_H5AD)

RAW input integrity verified: all 8 files are unchanged.
Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS1_GSE103224\GSE103224_all_samples_inner.h5ad
